# Appendix E: Lifespan Scaling — Verification Code

This notebook reproduces all calculations from Appendix E of the Fractal Universe manuscript.
Each section below corresponds to a named analysis in the appendix.
All calculations are reproducible with a fixed random seed.

## Setup: Imports and Configuration

In [ ]:
#!/usr/bin/env python3
"""
Appendix E: Complete verification code
All calculations reproducible with fixed seed.
"""

import numpy as np
from scipy import stats

np.random.seed(42)

## Biological Regression (12 species)

Verifies the biological scaling exponents from Section 1–2 of Appendix E:
all 12 species, excluding bacteria/yeast (10 species), and mammals only (7 species).

In [ ]:
bio_data = [
    ("E. coli",       1.0e-15,  20/525600),
    ("S. cerevisiae", 5.0e-14,  90/525600),
    ("Human cell",    1.0e-11,  14/365.25),
    ("C. elegans",    1.0e-9,   20/365.25),
    ("Mouse",         2.5e-2,   4.0),
    ("Rat",           3.0e-1,   5.0),
    ("Cat",           4.0,      30.0),
    ("Dog",           30.0,     29.0),
    ("Human",         70.0,     122.5),
    ("Elephant",      4000,     65.0),
    ("Bowhead whale", 1.0e5,    211.0),
    ("Giant tortoise", 250,     190.0),
]

names = [d[0] for d in bio_data]
masses = np.array([d[1] for d in bio_data])
lifespans = np.array([d[2] for d in bio_data])
logM = np.log10(masses)
logT = np.log10(lifespans)

# All species
r_all = stats.linregress(logM, logT)
print(f"All 12 species: n={r_all.slope:.4f}\u00b1{r_all.stderr:.4f}, R\u00b2={r_all.rvalue**2:.4f}")

# Excluding bacteria/yeast
mask = np.array([i >= 2 for i in range(12)])
r_10 = stats.linregress(logM[mask], logT[mask])
print(f"Excl bacteria/yeast (n=10): n={r_10.slope:.4f}\u00b1{r_10.stderr:.4f}, R\u00b2={r_10.rvalue**2:.4f}")

# Mammals only
mi = [4,5,6,7,8,9,10]
r_mam = stats.linregress(logM[mi], logT[mi])
print(f"Mammals only (n=7): n={r_mam.slope:.4f}\u00b1{r_mam.stderr:.4f}, R\u00b2={r_mam.rvalue**2:.4f}")

## Biology-Only Extrapolation (Section 5)

Tests whether the biological exponent alone can predict stellar lifespans.
Uses human reference (70 kg, 122.5 yr max lifespan) and sweeps exponents from 0.20 to 0.28.

In [ ]:
print("--- Biology-Only Cosmic Extrapolation ---")
M_ref, T_ref = 70.0, 122.5
for n_val in [0.20, 0.21, 0.22, 0.25, 0.28]:
    T_sun = T_ref * (1.989e30 / M_ref) ** n_val
    print(f"  n={n_val}: Sun = {T_sun:.2e} yr (actual: 1e10, ratio: {1e10/T_sun:.1f}\u00d7)")

## Cross-Scale Regression (Section 6)

Fits an OLS regression through Cell + Human + Sun, then predicts the Milky Way
galaxy lifespan as an out-of-sample test.

In [ ]:
print("--- Cross-Scale Regression ---")
cs_logM = np.array([np.log10(1e-12), np.log10(70), np.log10(1.989e30)])
cs_logT = np.array([np.log10(14/365.25), np.log10(80), np.log10(1e10)])
r_cs = stats.linregress(cs_logM, cs_logT)

print(f"Slope: {r_cs.slope:.4f} \u00b1 {r_cs.stderr:.4f}")
print(f"R\u00b2: {r_cs.rvalue**2:.4f}")
print(f"Kleiber distance: {abs(r_cs.slope - 0.25)/r_cs.stderr:.1f}\u03c3")

# Galaxy prediction
mw_logT = r_cs.intercept + r_cs.slope * np.log10(1.5e42)
print(f"MW prediction: 10^{mw_logT:.2f} = {10**mw_logT/1e12:.1f} Tyr")
print(f"Observed: 10-100 Tyr. Factor from lower: {10**mw_logT/1e13:.1f}\u00d7")

# Universe
uni_logT = r_cs.intercept + r_cs.slope * np.log10(1e53)
print(f"Universe prediction: 10^{uni_logT:.1f} (overshoot: ~{10**uni_logT/1e14:.0f}\u00d7)")

## Cell Lifespan Sensitivity

Tests how the cross-scale regression slope and galaxy prediction change
when different cell lifespan values are used (3 to 300 days).

In [ ]:
print("--- Cell Lifespan Sensitivity ---")
for days in [3, 14, 28, 120, 300]:
    lM = np.array([np.log10(1e-12), np.log10(70), np.log10(1.989e30)])
    lT = np.array([np.log10(days/365.25), np.log10(80), np.log10(1e10)])
    r = stats.linregress(lM, lT)
    pred = r.intercept + r.slope * np.log10(1.5e42)
    print(f"  {days:>3d} days: slope={r.slope:.4f}, MW=10^{pred:.2f} ({10**pred/1e12:.1f} Tyr)")

## Star Choice Sensitivity

Repeats the cross-scale regression replacing the Sun with 7 different stars
to test whether the result depends on the specific stellar data point chosen.

In [ ]:
print("--- Star Choice Sensitivity ---")
stars = [(0.1, 1e13), (0.7, 4.5e10), (1.0, 1e10), (1.5, 3e9),
         (2.0, 1.5e9), (10.0, 2e7), (60.0, 3e6)]
for m_sun, life in stars:
    mass_kg = m_sun * 1.989e30
    lM = np.array([np.log10(1e-12), np.log10(70), np.log10(mass_kg)])
    lT = np.array([np.log10(14/365.25), np.log10(80), np.log10(life)])
    r = stats.linregress(lM, lT)
    pred = r.intercept + r.slope * np.log10(1.5e42)
    bio = "\u2713" if 0.18 <= r.slope <= 0.30 else "\u2717"
    print(f"  {m_sun:>4.1f} M\u2609: slope={r.slope:.3f} {bio}, MW=10^{pred:.1f}")

## Energy Density

Compares the energy density (W/kg) of the Sun and a human,
showing that humans are ~5,938\u00d7 more energy-dense per kilogram.

In [ ]:
print("--- Energy Density ---")
sun_wpkg = 3.828e26 / 1.989e30
human_wpkg = 80 / 70
print(f"Sun: {sun_wpkg:.6f} W/kg")
print(f"Human: {human_wpkg:.4f} W/kg")
print(f"Ratio: {human_wpkg/sun_wpkg:,.0f}\u00d7")